# Multilingual Health Q&A — Starter Notebook

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

This notebook provides two baselines for the challenge:

| Baseline | Approach | Speed | Quality |
|---|---|---|---|
| **Baseline 1** | TF-IDF retrieval | Very fast | Low |
| **Baseline 2** | Multilingual LLM (mT5 / NLLB) | Moderate | Higher |

**Task:** Given a health question in one of five African languages (Amharic, Luganda, Akan, Swahili, or English), generate a fluent and accurate answer in the **same language**.

**Evaluation metrics:**
- `TargetRLF1` — ROUGE-L F1 score
- `TargetR1F1` — ROUGE-1 F1 score
- `TargetLLM`  — LLM-as-a-Judge score

> **Note:** All three submission columns should contain the same generated answer for each row. The platform computes all three metrics from that single answer.

In [15]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch

print('Packages installed successfully')


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Packages installed successfully



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print(' Imports complete')

 Imports complete


In [19]:
from pathlib import Path

# Point DATA_DIR directly to your unzipped folder
DATA_DIR = Path('./zindi_health_qa_data')

# Link the CSV files using the / operator
TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

# Output paths for each baseline (keeps them in the data folder, or you can change to '.')
OUTPUT_TFIDF = DATA_DIR / 'submission_tfidf_baseline.csv'
OUTPUT_LLM   = DATA_DIR / 'submission_llm_baseline.csv'

# Check if they exist now
for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {path}')

✅ zindi_health_qa_data\Train.csv
✅ zindi_health_qa_data\Test.csv
✅ zindi_health_qa_data\Val.csv
✅ zindi_health_qa_data\SampleSubmission.csv


In [20]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape             : {train.shape}')
print(f'Test shape              : {test.shape}')
print(f'Val shape               : {val.shape}')
print(f'Sample submission shape : {sample_submission.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Test columns :', test.columns.tolist())
print('Val columns  :', val.columns.tolist())

display(train.head(3))
display(test.head(3))
display(sample_submission.head(3))

Train shape             : (29815, 4)
Test shape              : (2618, 3)
Val shape               : (6686, 4)
Sample submission shape : (2618, 4)

Train columns: ['ID', 'input', 'output', 'subset']
Test columns : ['ID', 'input', 'subset']
Val columns  : ['ID', 'input', 'output', 'subset']


,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye w...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na w...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,"Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu taa de di dwuma, na yɛbɛyɛ dɛn ahwɛ ahu sɛ yɛde redi dwuma yiye?","Wɔ Ghana mu no, amanmmra no gye binary gender nkutoo tom a she/he edinnsiananmu nkutoo na ɛka ho",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔn ho fi ho anaa nna mu adwumadi a wɔtwentwɛn so no boa ma asiane so tew?,"Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔn nan ase a, ɛboa ma asiane nso tew denam asiane a ɛwɔ STI ne HIV a ɛb...",Aka_Gha


,ID,input,subset
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.",Aka_Gha
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,Aka_Gha
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,Aka_Gha


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
1,ID_TS_Aka_Gha_1C80317F,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
2,ID_TS_Aka_Gha_06671AD1,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"


In [21]:
# Explore language distribution in training data
print('Language distribution in training set:')
display(train['subset'].value_counts())

Language distribution in training set:


subset
Eng_Uga    7624
Aka_Gha    4455
Eng_Gha    4443
Eng_Eth    3915
Lug_Uga    3383
Eng_Ken    2080
Swa_Ken    2070
Amh_Eth    1845
Name: count, dtype: int64

In [22]:
ID_COL           = 'ID'
TEST_ID_COL      = 'ID'
QUESTION_COL     = 'input'
TEST_QUESTION_COL= 'input'
ANSWER_COL       = 'output'
LANG_COL         = 'subset'
TEST_LANG_COL    = 'subset'

print(f'  Train ID        : {ID_COL}')
print(f'  Test ID         : {TEST_ID_COL}')
print(f'  Train question  : {QUESTION_COL}')
print(f'  Test question   : {TEST_QUESTION_COL}')
print(f'  Train answer    : {ANSWER_COL}')
# ── Language code mapping ──────────────────────────────────────────────────────
# The `subset` column encodes language and country as "<LangCode>_<CountryCode>".
# Only the first part identifies the language; the second is the country.
# This dictionary maps the language prefix to a full language name used in prompts.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    """
    Extract the full language name from a subset code such as 'Amh_Eth' or 'Aka_Gha'.
    Falls back to the raw code if the prefix is not recognised.
    """
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    lang_prefix = subset_code.split('_')[0]
    return SUBSET_TO_LANGUAGE.get(lang_prefix, subset_code)

print('Language mapping:')
for code, name in SUBSET_TO_LANGUAGE.items():
    print(f'  {code}_* → {name}')


  Train ID        : ID
  Test ID         : ID
  Train question  : input
  Test question   : input
  Train answer    : output
Language mapping:
  Eng_* → English
  Aka_* → Akan
  Lug_* → Luganda
  Swa_* → Swahili
  Amh_* → Amharic


In [23]:
def clean_text(x):
    """Strip whitespace and handle null values."""
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

# Remove rows with empty questions or answers
train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[TEST_QUESTION_COL]  != ''].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)

print(f'Cleaned train shape : {train.shape}')
print(f'Cleaned test shape  : {test.shape}')
print(f'Cleaned val shape   : {val.shape}')

Cleaned train shape : (29814, 4)
Cleaned test shape  : (2618, 3)
Cleaned val shape   : (6686, 4)


In [24]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        """Whitespace tokeniser — language-agnostic and safe for African scripts."""
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    def compute_rouge(predictions, references):
        """
        Compute mean ROUGE-1 and ROUGE-L F1 scores.

        Parameters
        ----------
        predictions : list[str]
        references  : list[str]

        Returns
        -------
        dict with rouge1_f1 and rougeL_f1
        """
        scorer = rouge_scorer.RougeScorer(
            ['rouge1', 'rougeL'],
            tokenizer    = WhitespaceTokenizer(),
            use_stemmer  = False,
        )
        r1_scores, rl_scores = [], []

        for pred, ref in zip(predictions, references):
            score = scorer.score(str(ref), str(pred))
            r1_scores.append(score['rouge1'].fmeasure)
            rl_scores.append(score['rougeL'].fmeasure)

        return {
            'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
            'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
        }

    def compute_rouge_by_language(predictions, references, languages):
        """Compute ROUGE scores broken down by language."""
        results = {}
        lang_arr = np.array(languages)

        for lang in np.unique(lang_arr):
            mask    = lang_arr == lang
            preds_l = [p for p, m in zip(predictions, mask) if m]
            refs_l  = [r for r, m in zip(references,  mask) if m]
            results[lang] = compute_rouge(preds_l, refs_l)

        return pd.DataFrame(results).T

    print(' ROUGE scorer loaded')

except ImportError:
    print('rouge-score not installed. Run: pip install rouge-score')
    compute_rouge = None

 ROUGE scorer loaded


## 8 —Baseline 1: TF-IDF Retrieval

For each test question, find the most similar training question using TF-IDF character n-grams and return its answer.

**Why character n-grams?** Character-level features work across scripts (Latin, Amharic Ge'ez, etc.) without requiring language-specific tokenisation.


In [25]:
class TfidfRetrievalAnswerer:
    """
    TF-IDF nearest-neighbour retrieval baseline.

    Builds a per-language model if a group column is available,
    falling back to a global model for unseen groups.
    """

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        """Fit a vectoriser and nearest-neighbour index on a subset."""
        vectorizer = TfidfVectorizer(
            analyzer     = 'char_wb',
            ngram_range  = self.ngram_range,
            min_df       = 1,
            max_features = self.max_features,
            lowercase    = False,   # preserve case for non-Latin scripts
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers   = df[self.answer_col].fillna('').astype(str).tolist()

        X  = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)

        return {
            'vectorizer': vectorizer,
            'nn'        : nn,
            'answers'   : np.array(answers,   dtype=object),
            'questions' : np.array(questions, dtype=object),
        }

    def fit(self, df):
        """Fit the global model and per-group models."""
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        print(f'  Fitted global model + {len(self.models)} group model(s)')
        return self

    def _predict_one_from_model(self, question, model):
        Xq        = model['vectorizer'].transform([question])
        dist, idx = model['nn'].kneighbors(Xq, n_neighbors=1)
        i         = idx[0][0]
        sim       = 1 - float(dist[0][0])
        return model['answers'][i], sim, model['questions'][i]

    def predict_one(self, question, group=None):
        model = self.models.get(group, self.global_model) if group is not None else self.global_model
        return self._predict_one_from_model(question, model)

    def predict(self, df, question_col, group_col=None):
        outputs, similarities, matched = [], [], []
        for _, row in df.iterrows():
            question = clean_text(row[question_col])
            group    = row[group_col] if group_col and group_col in df.columns else None
            answer, sim, matched_q = self.predict_one(question, group)
            outputs.append(answer)
            similarities.append(sim)
            matched.append(matched_q)
        return outputs, similarities, matched

print('TfidfRetrievalAnswerer defined')

TfidfRetrievalAnswerer defined


In [26]:
# Choose grouping strategy — prefer config (language+country), else language
GROUP_COL      =  LANG_COL
TEST_GROUP_COL =  TEST_LANG_COL

print(f'Group column      : {GROUP_COL}')
print(f'Test group column : {TEST_GROUP_COL}')

Group column      : subset
Test group column : subset


In [27]:
# ── Validate TF-IDF baseline on the local validation set ──────────────────────
print('Training TF-IDF retrieval baseline on training partition...')

answerer_valid = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

valid_pred, valid_sim, valid_match = answerer_valid.predict(
    val,
    question_col = QUESTION_COL,
    group_col    = GROUP_COL,
)

if compute_rouge:
    metrics = compute_rouge(valid_pred, val[ANSWER_COL].tolist())
    print(f'\n TF-IDF Baseline — Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics["rougeL_f1"]:.4f}')

    # Break down by language
    if LANG_COL and LANG_COL in val.columns:
        print('\n ROUGE scores by language:')
        lang_metrics = compute_rouge_by_language(
            valid_pred,
            val[ANSWER_COL].tolist(),
            val[LANG_COL].tolist()
        )
        display(lang_metrics.round(4))

# Preview
preview = val[[ID_COL, QUESTION_COL, ANSWER_COL]].copy()
preview['baseline_answer']        = valid_pred
preview['retrieval_similarity']   = [f'{s:.3f}' for s in valid_sim]
preview['matched_train_question'] = valid_match
display(preview.head(5))

Training TF-IDF retrieval baseline on training partition...
  Fitted global model + 8 group model(s)

📊 TF-IDF Baseline — Validation ROUGE Scores
   ROUGE-1 F1 : 0.4212
   ROUGE-L F1 : 0.3660

📊 ROUGE scores by language:


,rouge1_f1,rougeL_f1
Aka_Gha,0.2832,0.1674
Amh_Eth,0.1455,0.1353
Eng_Eth,0.5215,0.5030
Eng_Gha,0.2582,0.1707
Eng_Ken,0.5989,0.5606
Eng_Uga,0.5169,0.4715
Lug_Uga,0.5155,0.4935
Swa_Ken,0.6031,0.5672


,ID,input,output,baseline_answer,retrieval_similarity,matched_train_question
0,ID_VL_Aka_Gha_A3B1799D,"Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn...",Nhyehyɛeɛ aa ama ne mu so te sɛ senea aborɔfo ka no 'STEM' ne 'vocational training' se ɛbɛ adrɛse mmabun kuokuo ahoh...,"Nkɔmmɔdi: Aban, NGO, ne amanaman ntam nnwumakuo ntam nkitahodiɛ bɛtumi ama nkitahodiɛ, nneɛma a wɔboaboa ano, ne dwu...",0.378,"Ɔkwan bɛn so na aban ahorow, ahyehyɛde ahorow a ɛnyɛ aban de (NGOs), ne amanaman ntam nnwumakuw ne fekubɔ betumi ahy..."
1,ID_VL_Aka_Gha_1C80317F,Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?,"Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abusuabɔ ho. Kamfo ...",Mmara kwan so hokwan ahorow kyerɛ hokwan ahorow ne ahobammɔ a mmara de ma ankorankoro. Wɔ nna ne awo akwahosan ho no...,0.545,"Dɛn ne mmara kwan so ahokwan ahorow, na dɛn nti na ɛho hia sɛ mmabun hu ho wɔ nna ne awoɔ akwahosan ho?"
2,ID_VL_Aka_Gha_06671AD1,"Mɛyɛ dɛn atumi abɔ asisifo ho amanneɛ wɔ ɔkwan a etu mpɔn na ahobammɔ wom so, na anammɔn bɛn na metumi atu de ahwɛ a...",Ayayade ho nsɛm a wɔbɛbɔ ho amanneɛ yiye na ahobammɔ wom no hwehwɛ sɛ wɔyɛ nneɛma a edidi so yi: Wɔkyerɛw ayayade no...,"Akwan ahorow: Siesie Wo Ho Di Kan: Kyerɛw wo nsɛmmisa ne wo dadwen to hɔ ansa na woakɔ. Di Nokware: Kasa fa wo nna, ...",0.292,Akwan bɛn so na metumi ne dɔkotafo anaa nɛɛse akasa afa me awoɔ mu apɔmuden ho na mahwɛ sɛ wɔate m’ahiadeɛ ne me dad...
3,ID_VL_Aka_Gha_BDD640FB,"Ɔkwan bɛn so na mmabun betumi de akwan ahorow a egyina nnipa hokwan ahorow so adi dwuma, de akamfo nna ho nkyerɛkyer...","Mmabun betumi de akwan a, egyina nnipa hokwan ahorow so adi dwuma de akamfo nna ho nkyerɛkyerɛ a edi mũ wɔ sukuu ne ...",Mmabun betumi akasa ama nna ho nkyerɛkyerɛ a edi mu wɔ wɔn sukuu ne mpɔtam hɔ denam: Akuo a sukuufo di anim a wɔde w...,0.642,Ɔkwan bɛn so na mmabun betumi akasa afa nna ho nkyerɛkyerɛ a edi mu wɔ wɔn sukuu ne mpɔtam hɔ?
4,ID_VL_Aka_Gha_46685257,"Dwuma bɛn na abɛɛfo mfidie 'Technology' di de boa mmabun ma wonya ɔhwɛ a ɛfata, a telehealth ne online resources ka ho?",Abɛɛfo mfidie te sɛ telehealth ma mmabun kwan ma wonya ɔhwɛ fi wɔn fie. Wei yi akwansidie te sɛ lore a wobɛfo ɛnam b...,Websites (nimdeɛ pa). Apps (de hwehwɛ nneɛma). SMS (nkɔmmɔbɔ mfidie). Telemedicine (intanɛt so ayaresa).,0.412,Sɛn na abɛɛfo mfidie 'technology' boa SRH nimdeɛ ho nkyerɛkyerɛ a ɛkɔ so wɔ wiase nyinaa?


In [28]:
# ── Train on all data and predict test answers
print('Training TF-IDF retrieval on full training set...')

answerer = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

test_pred_tfidf, test_sim, test_match = answerer.predict(
    test,
    question_col = TEST_QUESTION_COL,
    group_col    = TEST_GROUP_COL,
)

print(f'Generated {len(test_pred_tfidf)} predictions')

preview = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview['baseline_answer']      = test_pred_tfidf
preview['retrieval_similarity'] = [f'{s:.3f}' for s in test_sim]
display(preview.head(5))

Training TF-IDF retrieval on full training set...
  Fitted global model + 8 group model(s)
Generated 2618 predictions


,ID,input,baseline_answer,retrieval_similarity
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...",0.464
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...",0.451
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,0.250
3,ID_TS_Aka_Gha_BDD640FB,"Sɛnea amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu ka mmaabun nkitahodi ne ...","Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwumadiɛ a ɛma wɔn tumi yɛ adwuma, afotuo nhyehyɛeɛ, ne atipɛnfoɔ adesua n...",0.357
4,ID_TS_Aka_Gha_46685257,Adɛn nti na mmara nsesaeɛ 'policy advocacy' ho hia ma mmabun nyin wɔ biribiara mu ?,Kae sɛ obiara suahunu yɛ soronko. Dwen sɛ obiara wɔ n'adwene kyerɛ. Gye tom sɛ amammerɛ ne nyamesom sesa nkurɔfoɔ su...,0.361
